# Evaluating the Impact of Feature Selection on Machine Learning-Based Heart Disease Classification Using the UCI Cleveland Dataset

**MSc Dissertation — Analysis Notebook**

This notebook is organised to map directly onto a typical Chapter 3 (Methodology) / Chapter 4 (Results) structure:

1. Importing the dataset
2. Inspecting and understanding the data
3. Data-quality checks
4. Exploratory Data Analysis (EDA)
5. Data preprocessing
6. Train/test splitting
7. Baseline models (Logistic Regression, Random Forest, SVM) on the **full** feature set
8. Feature selection (filter, wrapper and embedded methods, with justification)
9. Models re-trained on the **selected** feature subset
10. Comparison of full vs. selected feature sets
11. Evaluation metrics (Accuracy, Precision, Recall/Sensitivity, Specificity, F1, ROC-AUC, Confusion Matrices)
12. Graphs and tables for the dissertation (saved as high-resolution figures)
13. Results exported to disk for direct use in Chapter 4
14. Analysis tied back to the research questions

> Run the cells **in order, top to bottom**. All figures and tables are written to an `outputs/` folder next to this notebook so you can drop them straight into your dissertation.


## 0. Setup: imports, reproducibility, output folders

In [ ]:
# --- Core libraries ---
import os
import warnings
import numpy as np
import pandas as pd

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing & model selection ---
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# --- Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# --- Feature selection ---
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif, RFECV

# --- Evaluation ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")

# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Plot style (consistent, print-friendly for a dissertation) ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

# --- Output folders (figures + tables for Chapter 4) ---
OUT_DIR = "outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

print("Setup complete. Figures ->", FIG_DIR, "| Tables ->", TAB_DIR)


## 1. Importing the dataset

The UCI Cleveland Heart Disease dataset (`processed_cleveland.data`) is a **headerless CSV** with 14 columns and missing values encoded as `"?"`. Column names follow the official UCI documentation (`heart-disease.names`).

> Update `DATA_PATH` below if your file lives somewhere else.


In [ ]:
DATA_PATH = "processed_cleveland.data"  # adjust path if needed, e.g. "data/processed_cleveland.data"

COLUMN_NAMES = [
    "age",       # age in years
    "sex",       # 1 = male, 0 = female
    "cp",        # chest pain type (1-4)
    "trestbps",  # resting blood pressure (mm Hg)
    "chol",      # serum cholesterol (mg/dl)
    "fbs",       # fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
    "restecg",   # resting ECG results (0-2)
    "thalach",   # maximum heart rate achieved
    "exang",     # exercise-induced angina (1 = yes, 0 = no)
    "oldpeak",   # ST depression induced by exercise relative to rest
    "slope",     # slope of the peak exercise ST segment (1-3)
    "ca",        # number of major vessels (0-3) colored by fluoroscopy
    "thal",      # 3 = normal, 6 = fixed defect, 7 = reversible defect
    "target"     # diagnosis of heart disease (0 = absent, 1-4 = present, increasing severity)
]

df = pd.read_csv(DATA_PATH, header=None, names=COLUMN_NAMES, na_values="?")

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 2. Inspecting and understanding the data

In [ ]:
df.info()


In [ ]:
print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)
print("\nFirst 5 rows:")
df.head()


In [ ]:
df.describe(include="all").T


## 3. Data-quality checks

### 3.1 Missing values


In [ ]:
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary["missing_count"] > 0]
print("Columns with missing values:")
missing_summary


### 3.2 Duplicate rows

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Number of fully duplicated rows: {n_duplicates}")
if n_duplicates > 0:
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)))


### 3.3 Incorrect / unusual values

Each variable has a documented valid range (UCI documentation). We check for values outside these ranges,
which would indicate data-entry errors rather than genuine biological extremes.


In [ ]:
valid_ranges = {
    "sex":      [0, 1],
    "cp":       [1, 2, 3, 4],
    "fbs":      [0, 1],
    "restecg":  [0, 1, 2],
    "exang":    [0, 1],
    "slope":    [1, 2, 3],
    "ca":       [0, 1, 2, 3],
    "thal":     [3, 6, 7],
    "target":   [0, 1, 2, 3, 4],
}

print("Checking categorical variables against documented valid categories:\n")
for col, valid_vals in valid_ranges.items():
    observed = set(df[col].dropna().unique())
    unexpected = observed - set(valid_vals)
    status = "OK" if not unexpected else f"UNEXPECTED VALUES: {unexpected}"
    print(f"  {col:10s} -> observed {sorted(observed)} | {status}")

print("\nChecking continuous variables for implausible values (e.g. <= 0 where physiologically impossible):\n")
continuous_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(df[continuous_cols].describe().T[["min", "max"]])


**Interpretation notes for Chapter 3 (fill in after running the cell above):**
- Confirm each categorical column only contains its documented codes.
- Watch for `trestbps` or `chol` values of 0 (physiologically implausible — often indicate recording errors) and decide how to treat them (this dataset typically has none, but always verify on your own run).
- `oldpeak` should be $\geq 0$.


### 3.4 Data types

In [ ]:
print(df.dtypes)
print("\nNote: several columns (sex, cp, fbs, restecg, exang, slope, ca, thal) are categorical/ordinal")
print("but are stored as float64 because of the '?' missing-value markers read by pandas.")
print("They will be explicitly cast after imputation in the preprocessing section.")


### 3.5 Class distribution (target variable)

In [ ]:
print("Original multiclass target distribution (0 = no disease, 1-4 = increasing severity):")
print(df["target"].value_counts().sort_index())

# The literature on this dataset (Detrano et al., 1989, and the vast majority of subsequent
# UCI Cleveland studies) treats this as a BINARY classification problem: presence (1) vs.
# absence (0) of heart disease, because classes 1-4 are clinically grouped as "disease present"
# and the multiclass split is heavily imbalanced (see counts above) with too few samples in
# classes 3 and 4 to model reliably. We therefore create a binary target for modelling, while
# keeping the original column for reference/EDA.
df["target_binary"] = (df["target"] > 0).astype(int)

print("\nBinary target distribution (0 = no disease, 1 = disease present):")
print(df["target_binary"].value_counts())
print(df["target_binary"].value_counts(normalize=True).round(3) * 100, "%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x="target", data=df, ax=axes[0], hue="target", palette="viridis", legend=False)
axes[0].set_title("Original class distribution (0-4)")
axes[0].set_xlabel("Diagnosis class")
axes[0].set_ylabel("Count")

sns.countplot(x="target_binary", data=df, ax=axes[1], hue="target_binary", palette="Set2", legend=False)
axes[1].set_title("Binary class distribution")
axes[1].set_xlabel("0 = No disease, 1 = Disease present")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_class_distribution.png"))
plt.show()


## 4. Exploratory Data Analysis (EDA)

### 4.1 Descriptive statistics (by class)


In [ ]:
desc_by_class = df.groupby("target_binary")[continuous_cols].agg(["mean", "std", "median", "min", "max"]).round(2)
desc_by_class.to_csv(os.path.join(TAB_DIR, "table_descriptive_stats_by_class.csv"))
desc_by_class


### 4.2 Distributions of continuous variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    sns.histplot(data=df, x=col, hue="target_binary", kde=True, ax=axes[i],
                 palette="Set1", alpha=0.5, element="step")
    axes[i].set_title(f"Distribution of {col}")

# hide the unused 6th subplot
axes[-1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "02_continuous_distributions.png"))
plt.show()


### 4.3 Distributions of categorical variables

In [ ]:
categorical_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, hue="target_binary", ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} vs target")
    axes[i].legend(title="Disease", labels=["No", "Yes"])

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_categorical_distributions.png"))
plt.show()


### 4.4 Boxplots — continuous variables by class (outlier inspection)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, col in enumerate(continuous_cols):
    sns.boxplot(data=df, x="target_binary", y=col, ax=axes[i], hue="target_binary",
                palette="Set2", legend=False)
    axes[i].set_xlabel("Disease (0/1)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "04_boxplots_continuous.png"))
plt.show()


### 4.5 Correlation analysis

Pearson correlation is appropriate for the continuous variables; we also include the encoded
categorical/ordinal variables and the binary target to see linear associations at a glance.
A more rigorous, model-agnostic view of feature relevance is given later via mutual information
(Section 8), since correlation only captures *linear* relationships.


In [ ]:
corr_cols = [c for c in df.columns if c not in ["target"]]  # keep target_binary, drop multiclass target
corr_matrix = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation matrix (Pearson)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_correlation_heatmap.png"))
plt.show()

corr_matrix.to_csv(os.path.join(TAB_DIR, "table_correlation_matrix.csv"))


In [ ]:
# Correlation of each feature with the binary target, sorted
target_corr = corr_matrix["target_binary"].drop("target_binary").sort_values(key=abs, ascending=False)
print("Features ranked by |correlation| with target_binary:\n")
print(target_corr.round(3))

plt.figure(figsize=(8, 6))
target_corr.plot(kind="barh", color=np.where(target_corr > 0, "firebrick", "steelblue"))
plt.title("Correlation of each feature with target_binary")
plt.xlabel("Pearson correlation coefficient")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "06_target_correlation_ranked.png"))
plt.show()


## 5. Data preprocessing

Steps applied, in order:
1. **Missing values** — `ca` (4 missing) and `thal` (2 missing) are imputed using the **mode**, since both
   are categorical/ordinal variables and the proportion missing is negligible (<2%), so mode imputation
   introduces minimal bias while avoiding the loss of otherwise complete rows.
2. **Encoding** — categorical variables in this dataset are already numerically coded by the UCI donors
   (e.g. `cp` 1–4, `thal` 3/6/7). Since tree-based and margin-based models can operate on these ordinal
   codes directly and the cardinality is low, we retain the integer encoding rather than one-hot encoding,
   which would inflate dimensionality for a dataset with only 303 rows. This is explicitly justified in
   the write-up as a deliberate methodological choice.
3. **Scaling** — continuous variables are standardised (zero mean, unit variance) because Logistic
   Regression and SVM are distance/gradient-based and sensitive to feature scale; Random Forest is
   scale-invariant but is included in the same pipeline for consistency and to avoid data leakage.


In [ ]:
# --- 5.1 Handle missing values ---
df_clean = df.copy()

for col in ["ca", "thal"]:
    mode_val = df_clean[col].mode()[0]
    n_missing = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f"Imputed {n_missing} missing value(s) in '{col}' with mode = {mode_val}")

# --- 5.2 Cast categorical/ordinal columns to int now that they are complete ---
categorical_int_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
df_clean[categorical_int_cols] = df_clean[categorical_int_cols].astype(int)

print("\nRemaining missing values after cleaning:", df_clean.isna().sum().sum())
df_clean.head()


In [ ]:
# --- 5.3 Define feature matrix X and target y (binary classification) ---
FEATURE_COLS = [c for c in df_clean.columns if c not in ["target", "target_binary"]]
X = df_clean[FEATURE_COLS].copy()
y = df_clean["target_binary"].copy()

print("Feature columns used for modelling:", FEATURE_COLS)
print("X shape:", X.shape, "| y shape:", y.shape)


## 6. Train/test splitting

A stratified 80/20 split preserves the class balance of `target_binary` in both sets, which matters
given the moderate class imbalance (≈54% vs. 46%). Scaling is fit **only on the training set** and
then applied to the test set, to prevent data leakage.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")
print("\nTrain class balance:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:\n", y_test.value_counts(normalize=True).round(3))

# Scale continuous features (fit on train only, transform both)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("\nScaling applied to continuous columns:", continuous_cols)


## 7. Baseline models (full feature set)

Three widely-used classifiers, chosen to span different learning paradigms — a linear model
(Logistic Regression), an ensemble of trees (Random Forest), and a margin-based kernel method (SVM) —
are trained on the **full** 13-feature set as the baseline against which feature selection will be
compared. A fixed `random_state` and 5-fold stratified cross-validation are used for fair comparison.


In [ ]:
def make_models():
    """Factory so we get fresh, identically-configured model instances each time."""
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        "SVM (RBF kernel)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    }

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name, feature_set_name):
    """Fit a model and compute the full evaluation-metric suite required for Chapter 4."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_te)

    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    results = {
        "Model": model_name,
        "Feature Set": feature_set_name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall (Sensitivity)": recall_score(y_te, y_pred),
        "Specificity": specificity,
        "F1-score": f1_score(y_te, y_pred),
        "ROC-AUC": roc_auc_score(y_te, y_proba),
    }
    return results, model, y_pred, y_proba

baseline_results = []
baseline_models_fitted = {}
baseline_preds = {}

for name, mdl in make_models().items():
    res, fitted, y_pred, y_proba = evaluate_model(
        mdl, X_train_scaled, y_train, X_test_scaled, y_test, name, "Full feature set"
    )
    baseline_results.append(res)
    baseline_models_fitted[name] = fitted
    baseline_preds[name] = (y_pred, y_proba)

baseline_results_df = pd.DataFrame(baseline_results).set_index("Model")
baseline_results_df.round(4)


In [ ]:
# 5-fold cross-validated accuracy on the training set, for robustness (not just a single split)
print("5-fold stratified cross-validation accuracy (training set, full feature set):\n")
for name, mdl in make_models().items():
    scores = cross_val_score(mdl, X_train_scaled, y_train, cv=cv_strategy, scoring="accuracy")
    print(f"  {name:22s}: mean = {scores.mean():.4f}  (+/- {scores.std():.4f})")


## 8. Feature selection

### Academic justification for the chosen techniques

To make the comparison robust and defensible in the dissertation, **three complementary categories**
of feature selection are applied and cross-checked, following the standard taxonomy in the feature
selection literature (Guyon & Elisseeff, 2003; Chandrashekar & Sahin, 2014):

| Category | Technique used | Rationale |
|---|---|---|
| **Filter** | Mutual Information (`mutual_info_classif`) and ANOVA F-test (`f_classif`) via `SelectKBest` | Model-agnostic, computationally cheap, captures both linear (F-test) and non-linear (MI) dependence between each feature and the target — appropriate as a first-pass screen before modelling. |
| **Wrapper** | Recursive Feature Elimination with cross-validation (`RFECV`), using Logistic Regression as the estimator | Evaluates feature subsets using actual downstream model performance (5-fold CV accuracy), directly optimising for the classification task rather than a proxy statistic. |
| **Embedded** | Random Forest feature importance (Gini importance) | Feature selection is a by-product of model training itself, capturing non-linear interactions the filter methods miss, and is a widely reported approach in Cleveland-dataset studies. |

The **final selected feature subset** used in Section 9 is the set of features chosen by RFECV
(wrapper method), since it directly optimises predictive performance via cross-validation rather
than relying on a univariate proxy statistic — this is stated explicitly as the methodological
decision for the dissertation. The filter and embedded results are retained and reported as
**triangulating evidence** (i.e., to show whether the different methods agree on which features
matter), which strengthens the discussion in Chapter 4/5.


### 8.1 Filter method 1 — ANOVA F-test

In [ ]:
f_scores, f_pvalues = f_classif(X_train_scaled, y_train)
anova_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "F-score": f_scores,
    "p-value": f_pvalues
}).sort_values("F-score", ascending=False).reset_index(drop=True)

anova_df.to_csv(os.path.join(TAB_DIR, "table_anova_f_scores.csv"), index=False)
anova_df.round(4)


### 8.2 Filter method 2 — Mutual Information

In [ ]:
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Mutual Information": mi_scores
}).sort_values("Mutual Information", ascending=False).reset_index(drop=True)

mi_df.to_csv(os.path.join(TAB_DIR, "table_mutual_information.csv"), index=False)
mi_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=anova_df, y="Feature", x="F-score", ax=axes[0], hue="Feature",
            palette="mako", legend=False)
axes[0].set_title("ANOVA F-test scores (higher = more discriminative)")

sns.barplot(data=mi_df, y="Feature", x="Mutual Information", ax=axes[1], hue="Feature",
            palette="rocket", legend=False)
axes[1].set_title("Mutual Information scores")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_filter_method_scores.png"))
plt.show()


### 8.3 Embedded method — Random Forest feature importance

In [ ]:
rf_for_importance = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf_for_importance.fit(X_train_scaled, y_train)

rf_importance_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": rf_for_importance.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

rf_importance_df.to_csv(os.path.join(TAB_DIR, "table_rf_feature_importance.csv"), index=False)

plt.figure(figsize=(8, 6))
sns.barplot(data=rf_importance_df, y="Feature", x="Importance", hue="Feature",
            palette="crest", legend=False)
plt.title("Random Forest feature importance (Gini)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "08_rf_feature_importance.png"))
plt.show()

rf_importance_df.round(4)


### 8.4 Wrapper method — Recursive Feature Elimination with Cross-Validation (RFECV)

RFECV recursively removes the weakest feature(s) and evaluates the remaining subset via cross-validated
accuracy, automatically selecting the subset size that maximises CV performance — this avoids having to
arbitrarily pre-specify "top-k".


In [ ]:
rfecv_estimator = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)

rfecv = RFECV(
    estimator=rfecv_estimator,
    step=1,
    cv=cv_strategy,
    scoring="accuracy",
    min_features_to_select=3,
)
rfecv.fit(X_train_scaled, y_train)

selected_features_rfecv = list(X_train_scaled.columns[rfecv.support_])
print(f"Optimal number of features selected by RFECV: {rfecv.n_features_}")
print(f"Selected features: {selected_features_rfecv}")

# Plot number of features vs. CV score
# (rfecv.cv_results_ also contains per-split 2D ranking/support arrays, so we pull out
#  only the 1-D summary fields needed for the plot rather than building a DataFrame from it)
n_features_range = rfecv.cv_results_["n_features"]
mean_scores = rfecv.cv_results_["mean_test_score"]

plt.figure(figsize=(8, 5))
plt.plot(n_features_range, mean_scores, marker="o")
plt.axvline(rfecv.n_features_, color="firebrick", linestyle="--",
            label=f"Optimal = {rfecv.n_features_} features")
plt.xlabel("Number of features selected")
plt.ylabel("Mean CV accuracy")
plt.title("RFECV: Cross-validated accuracy vs. number of features")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "09_rfecv_curve.png"))
plt.show()


### 8.5 Triangulating the three approaches

A quick cross-check: do the filter, embedded and wrapper methods broadly agree on which features
are most informative? This comparison itself is useful discussion material for Chapter 4/5.


In [ ]:
top_k = rfecv.n_features_

top_anova = set(anova_df.head(top_k)["Feature"])
top_mi = set(mi_df.head(top_k)["Feature"])
top_rf = set(rf_importance_df.head(top_k)["Feature"])
top_rfecv = set(selected_features_rfecv)

comparison_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "In ANOVA top-k": [f in top_anova for f in FEATURE_COLS],
    "In MI top-k": [f in top_mi for f in FEATURE_COLS],
    "In RF-importance top-k": [f in top_rf for f in FEATURE_COLS],
    "Selected by RFECV (final)": [f in top_rfecv for f in FEATURE_COLS],
})
comparison_df["Agreement count"] = comparison_df[[
    "In ANOVA top-k", "In MI top-k", "In RF-importance top-k", "Selected by RFECV (final)"
]].sum(axis=1)
comparison_df = comparison_df.sort_values("Agreement count", ascending=False).reset_index(drop=True)

comparison_df.to_csv(os.path.join(TAB_DIR, "table_feature_selection_triangulation.csv"), index=False)
comparison_df


In [ ]:
# --- Final selected feature subset used for Section 9 onward ---
SELECTED_FEATURES = selected_features_rfecv
print(f"FINAL SELECTED FEATURE SET (via RFECV, n={len(SELECTED_FEATURES)}):")
print(SELECTED_FEATURES)

X_train_selected = X_train_scaled[SELECTED_FEATURES]
X_test_selected = X_test_scaled[SELECTED_FEATURES]


## 9. Models using the selected feature subset

The same three classifiers, with the same hyperparameters as the baseline, are retrained using
**only** `SELECTED_FEATURES`. Keeping the model configuration identical isolates the effect of
feature selection as the sole experimental variable, which is essential for a clean comparison.


In [ ]:
selected_results = []
selected_models_fitted = {}
selected_preds = {}

for name, mdl in make_models().items():
    res, fitted, y_pred, y_proba = evaluate_model(
        mdl, X_train_selected, y_train, X_test_selected, y_test, name, "Selected features"
    )
    selected_results.append(res)
    selected_models_fitted[name] = fitted
    selected_preds[name] = (y_pred, y_proba)

selected_results_df = pd.DataFrame(selected_results).set_index("Model")
selected_results_df.round(4)


In [ ]:
print("5-fold stratified cross-validation accuracy (training set, SELECTED features):\n")
for name, mdl in make_models().items():
    scores = cross_val_score(mdl, X_train_selected, y_train, cv=cv_strategy, scoring="accuracy")
    print(f"  {name:22s}: mean = {scores.mean():.4f}  (+/- {scores.std():.4f})")


## 10. Comparison of full vs. selected feature sets

In [ ]:
all_results_df = pd.concat([
    pd.DataFrame(baseline_results),
    pd.DataFrame(selected_results)
], ignore_index=True)

all_results_df = all_results_df.set_index(["Model", "Feature Set"]).round(4)
all_results_df.to_csv(os.path.join(TAB_DIR, "table_full_vs_selected_comparison.csv"))
all_results_df


In [ ]:
# Reshape for a grouped bar chart: one panel per metric
metrics_to_plot = ["Accuracy", "Precision", "Recall (Sensitivity)", "Specificity", "F1-score", "ROC-AUC"]
plot_df = pd.concat([pd.DataFrame(baseline_results), pd.DataFrame(selected_results)], ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    sns.barplot(data=plot_df, x="Model", y=metric, hue="Feature Set", ax=axes[i], palette="Set1")
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis="x", rotation=20)
    if i != 0:
        axes[i].legend_.remove() if axes[i].get_legend() else None

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "10_full_vs_selected_all_metrics.png"))
plt.show()


In [ ]:
# Percentage-point change in each metric (Selected - Full) per model, for direct discussion in Chapter 4
delta_records = []
for model_name in baseline_results_df.index:
    row = {"Model": model_name}
    for metric in metrics_to_plot:
        full_val = baseline_results_df.loc[model_name, metric]
        sel_val = selected_results_df.loc[model_name, metric]
        row[f"Delta {metric}"] = round(sel_val - full_val, 4)
    delta_records.append(row)

delta_df = pd.DataFrame(delta_records).set_index("Model")
delta_df.to_csv(os.path.join(TAB_DIR, "table_metric_deltas_selected_minus_full.csv"))
print("Change in each metric when moving from the full feature set to the selected subset")
print("(positive = improvement after feature selection):\n")
delta_df


## 11. Detailed evaluation: confusion matrices and ROC curves

All headline metrics (Accuracy, Precision, Recall/Sensitivity, Specificity, F1, ROC-AUC) were already
computed in Sections 7, 9 and 10. This section adds the visual diagnostics dissertations typically
expect: confusion matrices and ROC curves for every model, on both feature sets.


In [ ]:
def plot_confusion_matrices(preds_dict, y_true, title_suffix, filename):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (name, (y_pred, _)) in zip(axes, preds_dict.items()):
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=["No disease", "Disease"], yticklabels=["No disease", "Disease"])
        ax.set_title(f"{name}\n{title_suffix}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename))
    plt.show()

plot_confusion_matrices(baseline_preds, y_test, "(Full feature set)", "11_confusion_matrices_full.png")
plot_confusion_matrices(selected_preds, y_test, "(Selected features)", "12_confusion_matrices_selected.png")


In [ ]:
def plot_roc_curves(preds_full, preds_selected, y_true, filename):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    for name, (_, y_proba) in preds_full.items():
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc_val = roc_auc_score(y_true, y_proba)
        axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")
    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[0].set_title("ROC curves — Full feature set")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend(loc="lower right")

    for name, (_, y_proba) in preds_selected.items():
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc_val = roc_auc_score(y_true, y_proba)
        axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[1].set_title("ROC curves — Selected features")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename))
    plt.show()

plot_roc_curves(baseline_preds, selected_preds, y_test, "13_roc_curves_comparison.png")


### Full `classification_report` for every model/feature-set combination (precision, recall, F1 per class)

In [ ]:
for feature_set_name, preds_dict in [("Full feature set", baseline_preds), ("Selected features", selected_preds)]:
    print("=" * 70)
    print(feature_set_name)
    print("=" * 70)
    for name, (y_pred, _) in preds_dict.items():
        print(f"\n--- {name} ---")
        print(classification_report(y_test, y_pred, target_names=["No disease", "Disease"], digits=3))


## 12. Summary graphs and tables for the dissertation

All figures have already been saved to `outputs/figures/` (300 dpi PNGs) and all tables to
`outputs/tables/` (CSV) as each section ran. This cell lists everything that was produced, as a
convenient checklist/appendix reference for Chapter 4.


In [ ]:
print("Figures saved to:", os.path.abspath(FIG_DIR))
for f in sorted(os.listdir(FIG_DIR)):
    print("  -", f)

print("\nTables saved to:", os.path.abspath(TAB_DIR))
for f in sorted(os.listdir(TAB_DIR)):
    print("  -", f)


## 13. Exporting consolidated results for Chapter 4

A single consolidated results file (Excel workbook, one sheet per table) is produced so you can
pull numbers directly into your write-up without hunting through multiple CSVs.


In [ ]:
summary_path = os.path.join(OUT_DIR, "dissertation_results_summary.xlsx")

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
    all_results_df.reset_index().to_excel(writer, sheet_name="Full_vs_Selected_Metrics", index=False)
    delta_df.reset_index().to_excel(writer, sheet_name="Metric_Deltas", index=False)
    anova_df.to_excel(writer, sheet_name="ANOVA_F_Test", index=False)
    mi_df.to_excel(writer, sheet_name="Mutual_Information", index=False)
    rf_importance_df.to_excel(writer, sheet_name="RF_Feature_Importance", index=False)
    comparison_df.to_excel(writer, sheet_name="Feature_Selection_Triangulation", index=False)
    desc_by_class.to_excel(writer, sheet_name="Descriptive_Stats_By_Class")
    corr_matrix.to_excel(writer, sheet_name="Correlation_Matrix")

print("Consolidated results workbook written to:", os.path.abspath(summary_path))


## 14. Analysis addressing the research questions

This section auto-generates a short, data-driven narrative from the results computed above. Use it as
a **first draft** for Chapter 4/5 discussion — refine the wording and add citations to your literature
review, but the numbers/claims are pulled directly and reproducibly from this run.

Typical research questions for this dissertation topic, and how the notebook answers them:

- **RQ1: Does feature selection improve (or maintain) classification performance while reducing dimensionality?**
  → Answered by the `delta_df` table in Section 10 and the printed narrative below.
- **RQ2: Which features are most predictive of heart disease in the Cleveland dataset, and do different feature-selection methods agree?**
  → Answered by the triangulation table in Section 8.5.
- **RQ3: Which algorithm (Logistic Regression, Random Forest, SVM) benefits most from feature selection?**
  → Answered by comparing per-model deltas below.


In [ ]:
print("=" * 78)
print("AUTO-GENERATED RESULTS NARRATIVE (edit/expand this for your write-up)")
print("=" * 78)

n_full = len(FEATURE_COLS)
n_sel = len(SELECTED_FEATURES)
print(f"\n1) DIMENSIONALITY REDUCTION\n"
      f"   Feature selection (RFECV, wrapped around Logistic Regression) reduced the feature\n"
      f"   space from {n_full} to {n_sel} features ({(1 - n_sel/n_full)*100:.1f}% reduction),\n"
      f"   retaining: {SELECTED_FEATURES}\n")

print("2) PERFORMANCE IMPACT PER MODEL (Selected - Full, in percentage points):")
for model_name in delta_df.index:
    acc_delta = delta_df.loc[model_name, "Delta Accuracy"] * 100
    auc_delta = delta_df.loc[model_name, "Delta ROC-AUC"] * 100
    direction_acc = "improved" if acc_delta > 0 else ("stayed the same" if acc_delta == 0 else "declined")
    print(f"   - {model_name}: Accuracy {direction_acc} by {acc_delta:+.2f} pp; "
          f"ROC-AUC changed by {auc_delta:+.2f} pp")

best_full_model = baseline_results_df["Accuracy"].idxmax()
best_selected_model = selected_results_df["Accuracy"].idxmax()
print(f"\n3) BEST-PERFORMING MODEL\n"
      f"   Full feature set   -> {best_full_model} "
      f"(Accuracy={baseline_results_df.loc[best_full_model, 'Accuracy']:.3f}, "
      f"ROC-AUC={baseline_results_df.loc[best_full_model, 'ROC-AUC']:.3f})\n"
      f"   Selected features  -> {best_selected_model} "
      f"(Accuracy={selected_results_df.loc[best_selected_model, 'Accuracy']:.3f}, "
      f"ROC-AUC={selected_results_df.loc[best_selected_model, 'ROC-AUC']:.3f})\n")

top_agreement = comparison_df.sort_values("Agreement count", ascending=False).head(5)
print("4) FEATURE IMPORTANCE CONSENSUS (top 5 features by cross-method agreement):")
for _, row in top_agreement.iterrows():
    print(f"   - {row['Feature']}: agreed upon by {row['Agreement count']}/4 methods")

print("\n" + "=" * 78)
print("Remember to: (a) discuss WHY these clinical features are plausible predictors")
print("(e.g. thal, cp, ca, oldpeak, thalach are established cardiology risk indicators),")
print("(b) relate the magnitude of any accuracy change to sample-size limitations (n=303),")
print("(c) discuss the clinical cost asymmetry between false negatives and false positives")
print("    (Sensitivity/Specificity trade-off) rather than relying on Accuracy alone.")
print("=" * 78)


---
### Notes on academic write-up

- **Chapter 3 (Methodology):** Sections 1, 3, 5, 6, 7, 8 map directly to "Data Collection", "Data
  Quality", "Preprocessing", "Experimental Design", "Baseline Models" and "Feature Selection
  Methodology" subsections respectively.
- **Chapter 4 (Results):** Sections 4 (EDA), 10 (comparison table + chart), 11 (confusion matrices,
  ROC curves, classification reports) supply the figures/tables; all are already saved at 300 dpi /
  as CSV in `outputs/`.
- **Chapter 5 (Discussion):** Use the auto-generated narrative in Section 14 as a skeleton, then
  connect each finding back to prior literature on the Cleveland dataset (Detrano et al., 1989;
  numerous ML benchmark papers using this dataset report baseline accuracies in the 80–88% range,
  which is a useful external benchmark to cite when interpreting your own results).
- All random seeds are fixed (`RANDOM_STATE = 42`) — report this in your methodology for reproducibility.
- Consider adding **hyperparameter tuning** (e.g. `GridSearchCV`, already imported above) as an
  extension if your dissertation scope allows it — this notebook deliberately keeps hyperparameters
  fixed across the full-vs-selected comparison so that feature selection is the *only* variable
  changing, which is the cleaner experimental design for directly answering your research questions.
